# Comparacao consolidada dos modelos

Este notebook consolida as predicoes de teste da baseline, Random Forest e MLP em arquivos comparaveis por estacao e data.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.modeling.gold_energy import configure_mlflow_tracking
from src.modeling.model_comparison import save_consolidated_model_comparison
from src.modeling.training_config import MLFLOW_EXPERIMENT_NAME

RUN_ARTIFACTS_DIR = configure_mlflow_tracking(PROJECT_ROOT, MLFLOW_EXPERIMENT_NAME)

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Diretorio de artefatos de modelagem: {RUN_ARTIFACTS_DIR}")

In [ ]:
# Gera os CSVs consolidados usando os arquivos de predicao ja salvos pelos notebooks anteriores.
comparison_outputs = save_consolidated_model_comparison(
    artifacts_dir=RUN_ARTIFACTS_DIR,
    project_root=PROJECT_ROOT,
    log_to_mlflow=True,
)

print(f"Consolidado wide: {comparison_outputs['wide_path']}")
print(f"Consolidado long: {comparison_outputs['long_path']}")
print(f"Resumo de metricas: {comparison_outputs['metrics_path']}")
print(f"Manifesto: {comparison_outputs['manifest_path']}")
print(f"Linhas wide: {comparison_outputs['wide_rows']}")
print(f"Linhas long: {comparison_outputs['long_rows']}")

In [ ]:
# Visualiza o resumo para conferir rapidamente qual modelo ficou melhor por grupo de metricas.
metrics_summary = comparison_outputs["metrics"].copy()
display(
    metrics_summary.sort_values(["metric_group", "target", "rmse"])
    .reset_index(drop=True)
)

display(
    metrics_summary[["model", "metric_group", "balanced_nrmse_group"]]
    .drop_duplicates()
    .sort_values(["metric_group", "balanced_nrmse_group"])
    .reset_index(drop=True)
)